<font size=10>**TASK 3 – CULINARY NETWORK ANALYSIS**</font> <a class="anchor" id='title'></a> 

**Bachelor's in Data Science - NOVA IMS (25/26)**

**Project**: *Straining the great southern Melting Pot*

**Group 8**
- Beatriz Marques 20231605
- David Carrilho 20231693
- Duarte Fernandes 20231619
- Filipe Caçador 20231707
- Mariana Calais-Pedro 20231641

**Notebook Description**:
This notebook implements a Named Entity Recognition (NER) pipeline to extract culinary entities (Dishes) and Cuisines from Atlanta restaurant reviews. We then construct a co-occurrence network to detect meaningful culinary clusters, analyzing how specific dishes define different cuisine types without relying on pre-existing labels.

<font>**IR3 – Dish Entity Networks and Cuisine Clusters**
 
Can we automatically extract dishes and related entities (e.g., locations) from restaurant reviews, and do the co-occurrence patterns of these entities form meaningful clusters that reveal cuisine types and cultural links? 

(NER + Co-occurrence Analysis + Clustering)</font>

<font color='#BFD72' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a> 
- [1. Imports](#1)
- [2. Data](#2)
- [3. Named Entity Rcognition](#3)
    - [3.1 Specific Data Preparation](#3_1)
    - [3.2 Model Implementation](#3_2)
        - [3.2.1 SpaCy Sequence Labeling (CRF)](#3_2_1)
            - [3.2.1.1 Model Load and Definition](#3_2_1_1)
            - [3.2.1.2 Model Implementation on Dataset](#3_2_1_2)
    - [3.3 Model Evaluation](#3_3)
        - [3.3.1 Build Graph](#3_3_1)
        - [3.3.2 Sparsify Graph](#3_3_2)
        - [3.3.3 Clustering and Naming](#3_3_3)
        - [3.3.4 Visualization Creation](#3_3_4)
        - [3.3.5 Graphical and Tabular Representations](#3_3_5)
    - [3.4 Answer the Question: Culinary Network Analysis](#3_4)

# <font color='#BFD72F' size=6>**1. Imports**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)

In [1]:
import warnings
%load_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')

In [2]:
#pip install louvain
#pip install pyvis
#pip install sentence_transformers
#pip install hf_xet
#pip install jinja2

In [3]:
import sys
import os
import pandas as pd
import spacy
from spacy import displacy
import sklearn_crfsuite
from sklearn_crfsuite import metrics
from sklearn.model_selection import train_test_split
import networkx as nx
from collections import defaultdict, Counter
from sklearn.cluster import AgglomerativeClustering
from sentence_transformers import SentenceTransformer
from pyvis.network import Network
import numpy as np
import community  # python-louvain
from jinja2 import Template
import pkgutil
import webbrowser

# Get the absolute path of the source_code folder
source_code_path = os.path.abspath('../source')

# Add the source_code folder to sys.path
if source_code_path not in sys.path:
    sys.path.append(source_code_path)

from my_utils import *
from visualizations import *
from general_preprocessing1 import *
from named_entity_recognition_graph_prep import *

# <font color='#BFD72F' size=6>**2. Data**</font> <a class="anchor" id="2"></a>
  
[Back to TOC](#toc)

In [4]:
dataset_original = load_dataset('../data/02_atlanta_restaurant_slice_2023_translated.csv')

In [5]:
dataset_original.head()

,title,categoryName,website,url,reviewsCount,stars,text,is_chain,total_reviews_by_title,latitude,longitude,num_sentences,00_before_translating_cleaning,lang_langdetect,lang_langid,needs_translation,text_translated,text_for_pipeline
0,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,"One word amazing!! The red fish, halibut, fr...",False,3349,33.779814,-84.410451,3,"One word amazing!! The red fish, halibut, frie...",en,en,False,"One word amazing!! The red fish, halibut, frie...","One word amazing!! The red fish, halibut, frie..."
1,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,First time here and the food is great and the ...,False,3349,33.779814,-84.410451,1,First time here and the food is great and the ...,en,en,False,First time here and the food is great and the ...,First time here and the food is great and the ...
2,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,I recently had the pleasure of dining at Optim...,False,3349,33.779814,-84.410451,14,I recently had the pleasure of dining at Optim...,en,en,False,I recently had the pleasure of dining at Optim...,I recently had the pleasure of dining at Optim...
3,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,Beautiful atmosphere and delicious food. All o...,False,3349,33.779814,-84.410451,9,Beautiful atmosphere and delicious food . All ...,en,en,False,Beautiful atmosphere and delicious food . All ...,Beautiful atmosphere and delicious food . All ...
4,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,We had a wonderful dinner at the Optimist. Our...,False,3349,33.779814,-84.410451,3,We had a wonderful dinner at the Optimist . Ou...,en,en,False,We had a wonderful dinner at the Optimist . Ou...,We had a wonderful dinner at the Optimist . Ou...


In [6]:
dataset_original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53565 entries, 0 to 53564
Data columns (total 18 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   title                           53565 non-null  object 
 1   categoryName                    53565 non-null  object 
 2   website                         50599 non-null  object 
 3   url                             53565 non-null  object 
 4   reviewsCount                    53565 non-null  int64  
 5   stars                           53565 non-null  float64
 6   text                            53565 non-null  object 
 7   is_chain                        53565 non-null  bool   
 8   total_reviews_by_title          53565 non-null  int64  
 9   latitude                        53565 non-null  float64
 10  longitude                       53565 non-null  float64
 11  num_sentences                   53565 non-null  int64  
 12  00_before_translating_cleaning  

| 🏷️ **Column Name** | 📝 **Description** |
|:-------------------|:-------------------|
|**title** | Name of the restaurant |
|**categoryName** | Labels that describe the restaurant's cuisine type |
|**website** | URL of the restaurant's webpage |
|**url** | URL of the restaurant's Google Maps page |
|**reviewsCount** | Total number of reviews for the restaurant at the time of scraping |
|**stars** | Customer rating (1 to 5) |
|**text** | Text of the review |

In [7]:
dataset = dataset_original.copy()

# <font color='#BFD72F' size=6>**3. Named Entity Recognition**</font> <a class="anchor" id="3"></a>
  
[Back to TOC](#toc)

## <font color='#BFD72F' size=6>3.1 Specific Data Preparation</font> <a class="anchor" id="3_1"></a>
  
[Back to TOC](#toc)

To prepare the raw review text for Named Entity Recognition (NER), the content is processed to extract detailed linguistic features, resulting in one primary column:

1. **`pos_list` & `tokens`** — linguistic processing (Tokenization + POS Tagging)
2. **`ner_labels_custom`** — target label alignment (BIO Tagging)

<font color='#BFD72F'>**Data Types & Outputs**</font>

| Column | Description |
|--------|-------------|
| **`pos_list`** | List of strings representing the Part-of-Speech tag for each token |
| **`tokens`** | List of individual strings (words/punctuation) per review |
| **`ner_labels_custom`** | List of BIO tags (e.g., B-dish, O) aligned 1-to-1 with tokens |


<font color='#BFD72F'>**1. Linguistic Feature Extraction (`pos_list`)**</font>

Each review is processed using the project’s `main_pipeline` (leveraging libraries like `spaCy` or `NLTK`), configured to **preserve**:

- **Casing** (critical for identifying Proper Nouns like "Italian" vs "italian")
- **Punctuation** (defines sentence boundaries)
- **Stopwords** (provides grammatical structure)

This produces **a structural representation of the text**, which is the foundational input for extracting features for the Conditional Random Field (CRF) model.

<font color='#BFD72F'>**2. NER Association (`ner_labels_custom`)**</font>

For later co-occurrence demonstrations, we must assign a NER label to every token. This is achieved via the `align_bio_to_custom_tokens` function, which:

1. **detects entities** using a base NLP model, which in this case is spaCy.
2. **maps** standard entity types (e.g., `ORG`, `GPE`) to our custom labels using an `equivalence_table`.
3. **aligns** these entities to our custom `tokens` list using the BIO Scheme:
    - **B-** (Beginning): The first token of an entity (e.g., "Spicy" in "Spicy Tuna").
    - **I-** (Inside): Subsequent tokens of the same entity (e.g., "Tuna").
    - **O** (Outside): Tokens that are not part of any entity.
---

<font color='#BFD72F'>**Why Preserve Casing and Punctuation?**</font>

Named Entity Recognition relies heavily on **orthographic** and **grammatical** cues:

- **Capitalization** is a strong indicator of Named Entities (e.g., "Mexican" vs. "mexican", "Burger King" vs. "king").
- **Punctuation** defines sentence boundaries and list structures (e.g., comma-separated ingredients).POS Tags disambiguate usage (e.g., "Order" as a Verb vs. "Order" as a Noun).
- **Over-normalization** (lowercasing everything) would strip away the visual cues that distinguish a specific Dish or Cuisine from common nouns.

<font color='#BFD72F'>**Benefits**</font>

- **Context Awareness**: The model learns from the surrounding words, not just the word itself.
- **Disambiguation**: POS tags help distinguish between identical words used differently (e.g., "Turkey" the country vs. "turkey" the food).
- **Granularity**: Token-level processing allows for the extraction of multi-word entities (e.g., "Spicy Tuna Roll").

<font color='#BFD72F'>**Limitations & Recommendations**</font>

- **Computational Cost**: POS tagging and NER extraction are significantly slower than simple text splitting. Use multiprocessing for large datasets.
- **Domain Adaptation**: Standard POS taggers are trained on news corpora; they may struggle with culinary slang or informal review text.
- **Validation**: Ensure that the tokenizer does not aggressively split entity names (e.g., keeping "T-Bone" as one unit if possible).

In [8]:
# Tokenization and POS tagging using project pipeline
dataset["pos_list"] =\
      dataset["text_for_pipeline"].map(lambda content : main_pipeline(content,
                                                            print_output=False,
                                                            no_stopwords=False,
                                                            lowercase=False,
                                                            lemmatized=False,
                                                            no_punctuation=False,
                                                            pos_tags_list='pos_list'
                                                            ))

dataset["tokens"] =\
      dataset["text_for_pipeline"].map(lambda content : main_pipeline(content,
                                                            print_output=False,
                                                            no_stopwords=False,
                                                            lowercase=False,
                                                            lemmatized=False,
                                                            no_punctuation=False,
                                                            tokenized_output=True
                                                            ))


In [9]:
# Sanity check
dataset[['text_for_pipeline','tokens', 'pos_list']].head()

,text_for_pipeline,tokens,pos_list
0,"One word amazing!! The red fish, halibut, frie...","[One, word, amazing, !, !, The, red, fish, ,, ...","[CD, NN, NN, ., ., DT, JJ, NN, ,, NN, ,, VBN, ..."
1,First time here and the food is great and the ...,"[First, time, here, and, the, food, is, great,...","[JJ, NN, RB, CC, DT, NN, VBZ, JJ, CC, DT, NN, ..."
2,I recently had the pleasure of dining at Optim...,"[I, recently, had, the, pleasure, of, dining, ...","[PRP, RB, VBD, DT, NN, IN, VBG, IN, NNP, IN, N..."
3,Beautiful atmosphere and delicious food . All ...,"[Beautiful, atmosphere, and, delicious, food, ...","[NNP, RB, CC, JJ, NN, ., DT, IN, DT, NN, NN, N..."
4,We had a wonderful dinner at the Optimist . Ou...,"[We, had, a, wonderful, dinner, at, the, Optim...","[PRP, VBD, DT, JJ, NN, IN, DT, NNP, ., PRP$, N..."


## <font color='#BFD72F' size=6>3.2 Model Implementation</font> <a class="anchor" id="3_2"></a>
  
[Back to TOC](#toc)

### <font color='#BFD72F' size=6>3.2.1 SpaCy Sequence Labeling</font> <a class="anchor" id="3_2_1"></a>
  
[Back to TOC](#toc)

#### **3.2.1.1 Model Load and Definition** <a class="anchor" id="3_2_1_1"></a>
  
[Back to TOC](#toc)

In [10]:
nlp = spacy.load("en_core_web_sm")

equivalence_table = {"PERSON":"-per",
                     "NORP":"-grp",
                     "FAC":"-fac",
                     "ORG":"-org",
                     "GPE":"-gpe",
                     "LOC":"-geo",
                     "DATE":"-date",
                     "TIME":"-tim",
                     "WORK_OF_ART":"-art",
                     "LAW":"-law",
                     "LANGUAGE":"-lang",
                     "EVENT":"eve",
                     "PRODUCT":"-prod",
                     "PERCENT":"-pct",
                     "MONEY":"-mon",
                     "QUANTITY":"-qty",
                     "CARDINAL":"-card",
                     "ORDINAL":"-ord",
                     "":""
                     }

#### **3.2.1.2 Model Implementation on Dataset** <a class="anchor" id="3_2_1_2"></a>
  
[Back to TOC](#toc)

In [11]:
dataset["ner_labels_custom"] = dataset.apply(
    lambda row: align_bio_to_custom_tokens(
        row["text_for_pipeline"], 
        row["tokens"],
        nlp,
        equivalence_table
    ),
    axis=1
)

dataset[['text_for_pipeline','tokens','ner_labels_custom']].head()

,text_for_pipeline,tokens,ner_labels_custom
0,"One word amazing!! The red fish, halibut, frie...","[One, word, amazing, !, !, The, red, fish, ,, ...","[B-card, O, O, O, O, O, O, O, O, O, O, O, O, O..."
1,First time here and the food is great and the ...,"[First, time, here, and, the, food, is, great,...","[B-ord, O, O, O, O, O, O, O, O, O, O, O, O, O]"
2,I recently had the pleasure of dining at Optim...,"[I, recently, had, the, pleasure, of, dining, ...","[O, O, O, O, O, O, O, O, B-per, O, B-gpe, O, B..."
3,Beautiful atmosphere and delicious food . All ...,"[Beautiful, atmosphere, and, delicious, food, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
4,We had a wonderful dinner at the Optimist . Ou...,"[We, had, a, wonderful, dinner, at, the, Optim...","[O, O, O, O, O, O, O, B-grp, O, O, O, O, B-car..."


## <font color='#BFD72F' size=6>3.3 Model Evaluation</font> <a class="anchor" id="3_3"></a>
  
[Back to TOC](#toc)

#### **3.3.1 Build Graph** <a class="anchor" id="3_3_1"></a>
  
[Back to TOC](#toc)

*To prevent inital graphical noise, this script first calculates global document frequencies for all entities, strictly ignoring generic stopwords like "food" or "place." It then executes a second pass to construct nodes only for entities that meet the MIN_NODE_FREQ threshold. Finally, it establishes weighted edges between these valid entities whenever they co-occur in the same review, representing the strength of their relationship.*

In [14]:
# We use a simple stoplist and min_freq to prevent the graph from being unreadable
STOPLIST = {"restaurant", "place", "spot", "dinner", "lunch", "menu", "food", "meal", "location"}
MIN_NODE_FREQ = 3  # Only show entities appearing at least 3 times

G_full = nx.Graph()

entity_document_frequency = Counter()

# A. Count Document Frequency
for idx, row in dataset.iterrows():
    ents = extract_entities(row["tokens"], row["ner_labels_custom"])
    
    # Create a set to hold unique entities found in *this specific* document
    seen_in_doc = set()
    
    for text_for_pipeline, tp in ents:
        etype = classify_entity(tp, location_included=False)
        if etype:
            text_norm = text_for_pipeline.lower()
            if text_norm not in STOPLIST:
                # Add to the set instead of the global counter immediately
                seen_in_doc.add(text_norm)
    
    # Update the global counter with the unique entities from this document
    entity_document_frequency.update(seen_in_doc)

# B. Build Edges (Co-occurrence)
for idx, row in dataset.iterrows():
    ents = extract_entities(row["tokens"], row["ner_labels_custom"])
    
    # Filter entities based on frequency and type
    filtered = []
    for text_for_pipeline, tp in ents:
        etype = classify_entity(tp, location_included=False)
        text_norm = text_for_pipeline.lower()
        
        if etype and text_norm not in STOPLIST and entity_document_frequency[text_norm] >= MIN_NODE_FREQ:
            filtered.append((text_norm, etype))

    # Add nodes
    for ent_text, ent_type in filtered:
        if not G_full.has_node(ent_text):
            G_full.add_node(ent_text, type=ent_type)

    # Add edges (Co-occurrence within the same row)
    for i in range(len(filtered)):
        for j in range(i+1, len(filtered)):
            a, _ = filtered[i]
            b, _ = filtered[j]
            
            # Simple weight increment
            if G_full.has_edge(a, b):
                G_full[a][b]['weight'] += 1
            else:
                G_full.add_edge(a, b, weight=1)

#### **3.3.2 Sparsify Graph** <a class="anchor" id="3_3_2"></a>
  
[Back to TOC](#toc)

*To improve graph readability, this function reduces density by retaining only the top k strongest connections for each node based on edge weight. It reconstructs the network using these significant links, effectively pruning weak associations while preserving the most important relationships. Finally, any nodes that become isolated during this filtering process are removed to ensure the visualization remains focused and further noise-free.*

In [15]:
# This keeps only the top-K strongest connections per node to reduce clutter
# Create the clean graph for visualization
G = sparsify_graph(G_full, k=5)

#### **3.3.3 Clustering and Naming** <a class="anchor" id="3_3_3"></a>
  
[Back to TOC](#toc)

*To uncover underlying patterns, the system applies the Louvain algorithm to partition the graph into distinct communities of tightly knit entities. For each resulting cluster, an automated naming function assigns a descriptive label by analyzing the weighted importance of its nodes, prioritizing broad categories like "Cuisine" over specific "Dishes". If no dominant cuisine is found, the cluster is named after its two most significant dish entities (e.g., "Sushi & Ramen Cluster").*

In [16]:
partition = community_detection(G)

clusters = defaultdict(list)
for node, cid in partition.items():
    clusters[cid].append(node)


# Generate names for all clusters
named_clusters = {
    cid: {
        "name": infer_cluster_name(nodes, G, location_included=False),
        "nodes": nodes
    }
    for cid, nodes in clusters.items()
}

#### **3.3.4 Visualization Creation** <a class="anchor" id="3_3_4"></a>
  
[Back to TOC](#toc)

*To visualize the results, this script generates an interactive HTML network using PyVis, where nodes are sized logarithmically by connectivity and color-coded by their assigned cluster. It visually distinguishes relationships by rendering the top 10% strongest edges in bold red, while keeping weaker connections faint to maintain readability. Finally, a ForceAtlas2 physics simulation is configured to arrange the graph organically, ensuring nodes are spaced out while keeping related clusters tight.*

In [17]:
net = Network(
    height="800px",
    width="100%",
    bgcolor="#ffffff",
    font_color="#333333",
    select_menu=True,
    filter_menu=True # Added filter menu
)

degrees = dict(G.degree(weight='weight'))
all_weights = [d['weight'] for u, v, d in G.edges(data=True)]
# Highlight top 10% strongest edges
strong_threshold = np.percentile(all_weights, 90) if all_weights else 0

print(f"Graph stats: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Highlighting edges with weight >= {strong_threshold}")

# Add Nodes
for node, data in G.nodes(data=True):
    ntype = data.get("type", "UNKNOWN")
    cluster_id = partition.get(node)
    
    group_name = named_clusters[cluster_id]["name"] if cluster_id in named_clusters else "Unknown"
    
    # Size calculation (log scale to prevent huge bubbles)
    size = 10 + np.log(degrees.get(node, 1) + 1) * 8
    
    title_html = f"<b>{node.title()}</b><br>Type: {ntype}<br>Conn: {degrees.get(node,0)}"

    net.add_node(
        node, 
        label=node, 
        title=title_html,
        value=size,
        group=group_name,
        borderWidth=1
    )

# Add Edges (Weak = Grey, Strong = Red)
edges_to_draw = []
for u, v, w in G.edges(data=True):
    weight = w['weight']
    is_strong = weight >= strong_threshold
    edges_to_draw.append((u, v, weight, is_strong))

# Sort so strong edges are drawn on top
edges_to_draw.sort(key=lambda x: x[2]) 

for u, v, weight, is_strong in edges_to_draw:
    if is_strong:
        net.add_edge(u, v, width=3, color={'color': '#e74c3c', 'opacity': 0.9})
    else:
        net.add_edge(u, v, width=1, color={'color': '#bdc3c7', 'opacity': 0.3})

# Physics Layout
net.set_options("""
{
  "physics": {
    "forceAtlas2Based": {
      "gravitationalConstant": -100,
      "centralGravity": 0.01,
      "springLength": 100,
      "springConstant": 0.08
    },
    "maxVelocity": 50,
    "solver": "forceAtlas2Based",
    "stabilization": {
      "enabled": true,
      "iterations": 1000
    }
  },
  "nodes": {
      "font": { "size": 14, "face": "tahoma" }
  }
}
""")

# Fix for PyVis template in some environments
try:
    data = pkgutil.get_data("pyvis", "templates/template.html")
    if data:
        Network.template = Template(data.decode("utf-8"))
except:
    pass # Fallback to default if template load fails

output_file = "entity_network_vis4.html"
net.write_html(output_file)
print(f"✔ Network saved to {output_file}")

Graph stats: 68 nodes, 113 edges
Highlighting edges with weight >= 4.0
✔ Network saved to entity_network_vis4.html


#### **3.3.5 Graphical and Tabular Representations** <a class="anchor" id="3_3_5"></a>
  
[Back to TOC](#toc)

In [18]:
cluster_data = []
for cid, nodes in clusters.items():
    name = infer_cluster_name(nodes, G, location_included=False)
    cluster_data.append({
        "Cluster ID": cid,
        "Inferred Name": name,
        "Entities": nodes,
        "Size": len(nodes)
    })

clusters_df = pd.DataFrame(cluster_data)

# Reorder columns for readability
clusters_df = clusters_df[["Cluster ID", "Inferred Name", "Size", "Entities"]]

clusters_df

,Cluster ID,Inferred Name,Size,Entities
0,0,Italian Cuisine,16,"[italian, risotto, cheese, roman, bolognese, a..."
1,2,Mexican Cuisine,18,"[mexican, texan, peruvian, fantastic, american..."
2,4,French Cuisine,10,"[french, spanish, pecan, togo, european, sean,..."
3,3,Chinese Cuisine,11,"[asian, english, swedish, thai, korean, salmon..."
4,1,Indian Cuisine,13,"[indian, garlic, british, vegetarian, caucasia..."


In [19]:
webbrowser.open("entity_network_vis4.html")

True

### <font color='#BFD72F' size=6>3.4. Answer the Question: Culinary Network Analysis </font> <a class="anchor" id="3_4"></a>
  
[Back to TOC](#toc)

Based on our analysis, we successfully extracted dishes and related entities to form meaningful semantic groups.

Therefore, we focus on the **co-occurrence patterns** of these entities—how often specific food items, cuisines, or locations appear in the same review—to uncover the underlying culinary structure of the dataset.

- **Cluster 0 (Italian)**: 16 entities (e.g., risotto, cheese, roman, bolognese)
- **Cluster 1 (Indian)**: 13 entities (e.g., garlic, british, vegetarian, south indian)
- **Cluster 2 (Mexican/American)**: 18 entities (e.g., texan, peruvian, fantastic, american)
- **Cluster 3 (Chinese)**: 11 entities (e.g., asian, thai, korean, salmon)
- **Cluster 4 (French/European)**: 10 entities (e.g., spanish, pecan, swiss, european)

While some noise exists (e.g., generic terms like "fantastic" appearing in specific clusters), the resulting network reflects a **clear semantic separation**:

> Keywords, such as ingredients, dishes or cultures, mentioned together in reviews naturally organize themselves into distinct "Cuisine Families" without manual labeling.

The network visualization supports this observation. The nodes are color-coded by their modularity class, showing distinct grouping where major cuisines (Italian, Indian, Chinese) act as central "hubs" that connect related ingredients and sub-types. This suggests that:

> The "Gravity" of major cuisine keywords is strong enough to pull related, specific dishes/ingredients (like risotto to Italian or garlic to Indian) into their correct orbit.

In summary, automated entity extraction combined with co-occurrence analysis offers a powerful method for structuring unstructured review text, making it a suitable choice for generating automatic tags or recommendation categories.

However, to improve the purity of clusters (e.g., ensuring "English" doesn't drift into the Chinese cluster), stricter stop-word filtering or dependency parsing to separate subject-object relationships may yield even cleaner definitions.